## SCOS Migration Output
**Source File**: `01_data_ingestion_schema.ipynb`  
**Migrated on**: 2026-04-28  
**Conversion Tool**: Snowpark Connect (SCOS)

### Changes Applied
- SparkSession creation replaced with `snowpark_connect.init_spark_session()`
- PySpark imports annotated with SCOS compatibility comments
- Incompatible patterns flagged with `# SCOS-EWI:` markers

### Known Limitations
- Review `# SCOS-EWI:` markers for manual conversion required


# 01 — Data Ingestion and Schema Enforcement
### Tasty Bytes Consulting — Financial Data Pipeline

**Stage 1 of 5** in the Tasty Bytes Consulting data pipeline.

This notebook ingests raw client, asset, and portfolio data, enforces
explicit schemas, handles nulls, and produces a data-quality summary
before passing clean DataFrames to downstream stages.

Demonstrates:
- `StructType` schema declaration and enforcement
- Type casting (string → date, decimal, boolean)
- Null handling strategies (`fillna`, `dropna`, `when`)
- Reject-row separation for bad source records
- Per-column null profiling

In [ ]:
import sys
sys.path.insert(0, '.')

# SCOS-EWI: [SPRKCNTPY1000] PySpark import — verify Snowpark Connect compatibility
from pyspark.sql import SparkSession
# SCOS-EWI: [SPRKCNTPY1000] PySpark import — verify Snowpark Connect compatibility
from pyspark.sql.functions import (
    col, when, lit, trim, lower, to_date,
    date_format, current_timestamp, sum as _sum,
)
# SCOS-EWI: [SPRKCNTPY1000] PySpark import — verify Snowpark Connect compatibility
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType,
    DecimalType, DateType, TimestampType, BooleanType,
)
# SCOS-EWI: [SPRKCNTPY1000] PySpark import — verify Snowpark Connect compatibility
from demo_data import create_spark_session, load_all, COMPANY_NAME
import snowpark_connect

# SCOS-EWI: [SPRKCNTPY1001] Session replaced with Snowpark Connect
spark = snowpark_connect.init_spark_session()
# SCOS-EWI: [SPRKCNTPY4000] setLogLevel removed — SparkContext not available in SCOS
print(f"Pipeline: {COMPANY_NAME}")
print(f"Spark version: {spark.version}")

## 1. Load shared financial data

All source DataFrames come from `demo_data.py` — the single source of truth
for Meridian Capital Advisors' reference data.

In [ ]:
data = load_all(spark)

raw_clients_df    = data["clients"]
raw_assets_df     = data["assets"]
raw_portfolios_df = data["portfolios"]

print("=== Raw record counts ===")
for name in ["clients", "assets", "portfolios"]:
    print(f"  {name:12s}: {data[name].count()} rows")

print("\n=== Client schema ===")
raw_clients_df.printSchema()

## 2. Type casting and coercion

Source strings are parsed to their target types in a single transformation
pass.  Bad casts become `null` and are handled in the next step.

In [ ]:
# Cast clients: parse date, round AUM to 2 dp, normalise strings
clients_typed = (
    raw_clients_df
    .withColumn("onboarded_date", to_date(col("onboarded_date"), "yyyy-MM-dd"))
    .withColumn("aum_usd",        col("aum_usd").cast(DecimalType(18, 2)))
    .withColumn("client_type",    trim(lower(col("client_type"))))
    .withColumn("risk_profile",   trim(lower(col("risk_profile"))))
    .withColumn("region",         trim(lower(col("region"))))
    .withColumn("etl_loaded_at",  current_timestamp())
)

# Cast portfolios: parse date, round NAV
portfolios_typed = (
    raw_portfolios_df
    .withColumn("inception_date", to_date(col("inception_date"), "yyyy-MM-dd"))
    .withColumn("nav_usd",        col("nav_usd").cast(DecimalType(18, 2)))
    .withColumn("strategy",       trim(lower(col("strategy"))))
    .withColumn("etl_loaded_at",  current_timestamp())
)

print("=== Clients after type casting ===")
clients_typed.select(
    "client_id", "client_name", "client_type", "risk_profile",
    "aum_usd", "onboarded_date", "etl_loaded_at"
).show(5, truncate=False)
clients_typed.printSchema()

## 3. Null handling strategies

Three approaches are used depending on the column's business criticality:
1. `fillna` — safe default for non-critical optional fields
2. `dropna` — remove records missing business-critical identifiers
3. `when / otherwise` — conditional derivation for enrichment columns

In [ ]:
# Strategy 1 — fill optional fields with safe defaults
clients_filled = clients_typed.fillna({
    "relationship_manager": "Unassigned",
    "region":               "unknown",
})

# Strategy 2 — drop records missing business keys
clients_clean = clients_filled.dropna(subset=["client_id", "client_name", "aum_usd"])
print(f"Rows after dropna on critical fields: {clients_clean.count()} (was {clients_filled.count()})")

# Strategy 3 — derive AUM tier from aum_usd
clients_enriched = clients_clean.withColumn(
    "aum_tier",
    when(col("aum_usd") < 20_000_000,    "Tier 3: <$20M")
    .when(col("aum_usd") < 100_000_000,  "Tier 2: $20-100M")
    .when(col("aum_usd") < 300_000_000,  "Tier 1: $100-300M")
    .otherwise(                           "Tier 0: $300M+")
)

print("\n=== Clients with AUM tier ===")
clients_enriched.select(
    "client_id", "client_name", "client_type", "aum_usd", "aum_tier"
).show(truncate=False)

## 4. Schema enforcement on incoming raw data

Simulates a realistic scenario: raw portfolio data arriving as strings
from an upstream CSV feed.  Invalid casts become `null`; records are
split into valid vs. rejected sets.

In [ ]:
# Simulate a raw CSV feed — all columns arrive as strings, some rows are malformed
raw_feed = [
    ("PRT-013", "CLT-003", "Harrington Tax Exempt", "income",   "Muni Bond Index", "USD", "2024-03-01", "8500000.00",  "true"),
    ("PRT-014", "CLT-008", "Chen Tech Focus",        "growth",   "NASDAQ 100",      "USD", "2024-05-15", "12000000.00", "true"),
    ("PRT-015", "CLT-999", "Orphan Portfolio",       "balanced", "60/40 Blend",     "USD", "not-a-date", "INVALID",     "false"),  # bad row
    ("PRT-016", "CLT-010", "Davidson Growth Sleeve", "growth",   "S&P 500",         "USD", "2024-06-01", "25000000.00", "true"),
]
raw_feed_schema = StructType([
    StructField(f, StringType(), True)
    for f in ["portfolio_id","client_id","portfolio_name","strategy",
              "benchmark","base_currency","inception_date","nav_usd","is_active"]
])
raw_feed_df = spark.createDataFrame(raw_feed, schema=raw_feed_schema)

# Apply target types — bad casts silently become null
coerced_df = (
    raw_feed_df
    .withColumn("inception_date", to_date(col("inception_date"), "yyyy-MM-dd"))
    .withColumn("nav_usd",        col("nav_usd").cast(DoubleType()))
    .withColumn("is_active",      col("is_active").cast(BooleanType()))
)

valid_df  = coerced_df.dropna(subset=["inception_date", "nav_usd"])
reject_df = coerced_df.filter(
    col("inception_date").isNull() | col("nav_usd").isNull()
)

print(f"Valid rows: {valid_df.count()},  Rejected rows: {reject_df.count()}")
print("\n=== Rejected records ===")
reject_df.show(truncate=False)

## 5. Data quality summary

Before passing data downstream, generate a per-column null count and
basic descriptive statistics for numerical fields.

In [ ]:
def null_summary(df, label: str = ""):
    """Print null count and null % for every column in *df*."""
    total = df.count()
    null_counts = df.select([
        _sum(col(c).isNull().cast(IntegerType())).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()

    rows = [
        (c, null_counts[c], round(null_counts[c] / total * 100, 1))
        for c in df.columns
    ]
    result = spark.createDataFrame(rows, ["column", "null_count", "null_pct"])
    if label:
        print(f"=== Null summary: {label} ===")
    result.show()

# SCOS-EWI: [SPRKCNTPY1000] PySpark import — verify Snowpark Connect compatibility
from pyspark.sql.types import IntegerType as _IT

null_summary(clients_enriched, "dim_clients")
null_summary(portfolios_typed, "dim_portfolios")

print("=== Descriptive statistics: clients ===")
clients_enriched.select("aum_usd").describe().show()

print("=== AUM distribution by tier ===")
clients_enriched.groupBy("aum_tier").count().orderBy("aum_tier").show()

## 6. Stage output — clean DataFrames ready for downstream use

Register clean DataFrames as temp views so subsequent pipeline stages
(or SQL queries) can access them.

In [ ]:
clients_enriched.createOrReplaceTempView("stg_clients")
# SCOS: createOrReplaceTempView supported in Snowpark Connect
portfolios_typed.createOrReplaceTempView("stg_portfolios")
# SCOS: createOrReplaceTempView supported in Snowpark Connect
raw_assets_df.createOrReplaceTempView("stg_assets")
# SCOS: createOrReplaceTempView supported in Snowpark Connect

print("Stage 1 complete — views registered: stg_clients, stg_portfolios, stg_assets")
print(f"  stg_clients    : {clients_enriched.count()} rows")
print(f"  stg_portfolios : {portfolios_typed.count()} rows")
print(f"  stg_assets     : {raw_assets_df.count()} rows")

# Quick cross-check: every portfolio should have a matching client
orphaned = spark.sql("""
    SELECT p.portfolio_id, p.client_id
    FROM   stg_portfolios p
    LEFT JOIN stg_clients c ON p.client_id = c.client_id
    WHERE  c.client_id IS NULL
""")
print(f"\nOrphaned portfolios (no matching client): {orphaned.count()}")

In [ ]:
# spark.stop()  # SCOS: session lifecycle managed by Snowpark Connect